# Network Intrusion Detection — Progressive Dataset Evaluation

**Paper:** Chua & Salam (2023), *Evaluation of ML Algorithms in Network-Based Intrusion Detection Using Progressive Dataset*, Symmetry 15, 1251

**Setup:** Run this header cell first every time you open a new Colab session.

In [ ]:
# ── Header cell: run this first in every new Colab session ──────────────────
import sys, os

# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Clone repo so src/ modules are importable
REPO_URL = 'https://github.com/Rosette28/data-science-cyber-final-project'  # ← update
REPO_DIR = '/content/ids-project'
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# 3. Install dependencies
!pip install -q -r {REPO_DIR}/requirements.txt

print('Environment ready.')

In [ ]:
# ── Global configuration — only line you change between runs ─────────────────
DATA_DIR = '/content/drive/MyDrive/ids_data/raw/'  # ← set to your Drive folder

SEED = 42
SUBSAMPLE_FRAC = 0.10  # 10% of each day's CSV, read at load time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from joblib import dump, load

np.random.seed(SEED)
pd.set_option('display.max_columns', 50)
sns.set_theme(style='whitegrid', palette='tab10')

print(f'DATA_DIR = {DATA_DIR}')

---
## §1 — Data Loading & Initial Inspection

Load CIC-IDS2017 (train) and CSE-CIC-IDS2018 (progressive test), keeping ~10% via chunked reading. Inspect shape, dtypes, memory usage, column names, and temporal structure.

In [ ]:
from src.data_loading import load_cic2017, load_cic2018, align_schemas

df_train = load_cic2017(DATA_DIR, subsample_frac=SUBSAMPLE_FRAC, seed=SEED)
df_test  = load_cic2018(DATA_DIR, subsample_frac=SUBSAMPLE_FRAC, seed=SEED)
df_train, df_test = align_schemas(df_train, df_test)

print('Train shape:', df_train.shape)
print('Test  shape:', df_test.shape)

### Shape, dtypes, memory, and column name inspection

In [ ]:
# ── Shape and memory ──────────────────────────────────────────────────────────
for name, df in [('Train (CIC-IDS2017)', df_train), ('Test  (CSE-CIC-IDS2018)', df_test)]:
    mem_mb = df.memory_usage(deep=True).sum() / 1e6
    print(f"{name}: {df.shape[0]:>7,} rows × {df.shape[1]} cols  |  {mem_mb:.1f} MB")

print()

# ── Dtype breakdown ───────────────────────────────────────────────────────────
print("Train dtype counts:")
print(df_train.dtypes.value_counts().to_string())
print("\nTest dtype counts:")
print(df_test.dtypes.value_counts().to_string())

In [ ]:
# ── Column name analysis ──────────────────────────────────────────────────────
# Features fall into five semantic groups derived from CICFlowMeter's documentation.
feature_groups = {
    'Packet length stats':    [c for c in df_train.columns if 'Packet Length' in c or 'Pkt Len' in c or 'Packet Size' in c or 'Segment Size' in c],
    'Packet counts / rates':  [c for c in df_train.columns if 'Packets' in c or 'Pkts' in c or 'Bytes' in c or 'Bulk' in c or 'Subflow' in c],
    'Inter-arrival times':    [c for c in df_train.columns if 'IAT' in c or 'Flow Duration' in c],
    'TCP flags':              [c for c in df_train.columns if 'Flag' in c or 'Win' in c],
    'Other / port / misc':    [c for c in df_train.columns if c not in sum([
        [c for c in df_train.columns if 'Packet Length' in c or 'Pkt Len' in c or 'Packet Size' in c or 'Segment Size' in c],
        [c for c in df_train.columns if 'Packets' in c or 'Pkts' in c or 'Bytes' in c or 'Bulk' in c or 'Subflow' in c],
        [c for c in df_train.columns if 'IAT' in c or 'Flow Duration' in c],
        [c for c in df_train.columns if 'Flag' in c or 'Win' in c],
        ['Label']
    ], []) and c != 'Label'],
}

print("Feature groups (76 features total after schema alignment):\n")
for group, cols in feature_groups.items():
    print(f"  {group} ({len(cols)}): {cols}")

print(f"\nLabel column: 'Label' — unique values in train: {df_train['Label'].unique()}")
print(f"                       — unique values in test:  {df_test['Label'].unique()}")

**Shape and memory** (after load + schema alignment, before deduplication):
- Train (CIC-IDS2017, 10% sample): ~281k rows × 77 cols, ~160 MB
- Test (CSE-CIC-IDS2018, 10% sample): ~125k rows × 77 cols, ~71 MB
- After hygiene cleaning (§1.3): **266,739 × 67 train / 106,906 × 67 test, 66 features**
- Both sets fit comfortably in free Colab RAM (~12 GB limit).

**Dtypes:** 52 `int64` (flag counts, raw packet/byte totals) · 24 `float32` (rates, means, stds — downcast from float64 to halve memory) · 1 `object` (Label).

**66 features across 5 semantic groups:**
- **Packet length stats:** min/max/mean/std of forward and backward packet sizes. Captures *what is being sent* — flooding attacks use fixed-size packets; exfiltration sends large payloads.
- **Packet counts / byte rates:** total packet counts, flow rates (bytes/s, packets/s), subflow counts. Captures *volume and asymmetry* — DoS shows extreme forward rate with near-zero backward.
- **Inter-arrival times (IAT):** min/max/mean/std of gaps between packets, plus Flow Duration. Captures *timing* — scans and floods have unusually small or regular IATs.
- **TCP flags:** SYN/FIN/RST/PSH/ACK/URG/ECE counts, initial TCP window sizes. Captures *connection behaviour* — SYN without ACK = SYN flood; window size fingerprints OS and botnet clients.
- **Other:** destination port, header lengths, active/idle time stats. Port alone is a strong discriminator — scanning generates traffic across unusual high ports.

**Labels:** 15 attack types + BENIGN in train; 10 attack types + BENIGN in test (2018 used integer encoding; 2017 had UTF-8 artifacts in Web Attack labels — both fixed at load time).

**Timestamp excluded from features:** Including raw calendar timestamps would cause time leakage ('2017 flow = benign, 2018 flow = attack'). IAT and Duration capture structural flow timing, not calendar position.


### Temporal structure and the progressive evaluation design

In [ ]:
import os, glob

# ── Per-day file breakdown for CIC-IDS2017 ────────────────────────────────────
cic2017_dir = os.path.join(DATA_DIR, 'cic2017')
cic2018_dir = os.path.join(DATA_DIR, 'cic2018')

print("CIC-IDS2017 source files (training set):")
for f in sorted(glob.glob(os.path.join(cic2017_dir, '*.csv'))):
    print(f"  {os.path.basename(f)}")

print("\nCSE-CIC-IDS2018 source files (progressive test set):")
for f in sorted(glob.glob(os.path.join(cic2018_dir, '*.csv'))):
    print(f"  {os.path.basename(f)}")

In [ ]:
# ── Attack type coverage in each dataset ─────────────────────────────────────
print("Attack types in TRAINING set (CIC-IDS2017):")
train_labels = df_train['Label'].value_counts()
print(train_labels.to_string())

print("\nAttack types in TEST set (CSE-CIC-IDS2018):")
test_labels = df_test['Label'].value_counts()
print(test_labels.to_string())

# ── Semantic grouping — map both datasets to a shared attack family ───────────
# The two datasets use different naming conventions for the same attack families.
# We normalise to a common family name before comparing.
_FAMILY = {
    # 2017 names
    'BENIGN':                    'Benign',
    'Bot':                       'Bot',
    'DDoS':                      'DDoS',
    'PortScan':                  'PortScan',
    'FTP-Patator':               'Brute Force - FTP',
    'SSH-Patator':               'Brute Force - SSH',
    'DoS slowloris':             'DoS - Slowloris',
    'DoS Slowhttptest':          'DoS - SlowHTTPTest',
    'DoS Hulk':                  'DoS - Hulk',
    'DoS GoldenEye':             'DoS - GoldenEye',
    'Heartbleed':                'Heartbleed',
    'Infiltration':              'Infiltration',
    'Web Attack - Brute Force':  'Brute Force - Web',
    'Web Attack - XSS':          'Brute Force - XSS',
    'Web Attack - Sql Injection':'SQL Injection',
    # 2018 names
    'Brute Force - Web':         'Brute Force - Web',
    'Brute Force - XSS':         'Brute Force - XSS',
    'DDoS - HOIC':               'DDoS',
    'DDoS - LOIC-UDP':           'DDoS',
    'DDoS - LOIC-HTTP':          'DDoS',
    'DoS - GoldenEye':           'DoS - GoldenEye',
    'DoS - Hulk':                'DoS - Hulk',
    'DoS - SlowHTTPTest':        'DoS - SlowHTTPTest',
    'DoS - Slowloris':           'DoS - Slowloris',
}

train_families = set(df_train['Label'].map(_FAMILY).dropna().unique())
test_families  = set(df_test['Label'].map(_FAMILY).dropna().unique())

in_both     = train_families & test_families
only_train  = train_families - test_families
only_test   = test_families  - train_families

print(f"\nAttack families in BOTH datasets:        {sorted(in_both)}")
print(f"Attack families ONLY in train (2017):    {sorted(only_train)}")
print(f"Attack families ONLY in test  (2018):    {sorted(only_test)}")

**Temporal structure:**

**Within CIC-IDS2017 (8 CSVs, Mon 3 Jul – Fri 7 Jul 2017):**
Monday = benign-only background. Attacks escalate day by day: brute-force (Tue) → DoS/DDoS (Wed) → Web Attacks + Infiltration (Thu) → DDoS + PortScan (Fri).

**Between datasets (~8-month gap):**
All 2017 data precedes all 2018 data — zero temporal overlap. This is the paper's core methodology: train on July 2017, evaluate on Feb–Mar 2018 traffic the model has never seen.

**Class imbalance:** both datasets are ~80-84% benign (actual: 81.9% train, 84.1% test). The authors address this by downsampling to 1:1 — discussed in §2.1 and §2.8.

**Attack family coverage — key insight:**
After normalising naming differences (e.g. 'DoS Hulk' ↔ 'DoS - Hulk', 'DDoS' ↔ 'DDoS - HOIC/LOIC'), the 2018 test set introduces **no genuinely new attack family**. Models are not facing unknown attacks — they face the same families through different tools in a changed environment.

**What actually causes the cross-dataset performance drop:**
- **Different tools, same family:** 2017's 'DDoS' and 2018's LOIC/HOIC produce different flow signatures (packet rates, sizes, timing) despite identical attack intent.
- **Shifted class frequencies:** Web Attack - XSS had 70 training samples; Brute Force - XSS is the dominant attack class in test (10,489 rows). A model that barely saw this pattern cannot classify it reliably at high volume.
- **Changed environment:** different machines, topology, and background benign traffic distribution.

This is **concept drift** (the data-generating process changed between 2017 and 2018), not classical overfitting (train-vs-held-out gap on the *same* distribution). We test this directly in Phase 8.1.


### Data hygiene scan

In [ ]:
# ── Duplicate rows ────────────────────────────────────────────────────────────
feat_cols = [c for c in df_train.columns if c != 'Label']

train_dups = df_train.duplicated(subset=feat_cols).sum()
test_dups  = df_test.duplicated(subset=feat_cols).sum()
print(f"Duplicate feature rows — train: {train_dups:,}  |  test: {test_dups:,}")

# ── Remaining NaN / inf (should be zero after _clean) ────────────────────────
train_nan = df_train[feat_cols].isnull().sum().sum()
test_nan  = df_test[feat_cols].isnull().sum().sum()
print(f"Remaining NaN values  — train: {train_nan}  |  test: {test_nan}")

train_inf = np.isinf(df_train[feat_cols].values).sum()
test_inf  = np.isinf(df_test[feat_cols].values).sum()
print(f"Remaining inf values  — train: {train_inf}  |  test: {test_inf}")

In [ ]:
# ── Constant / near-constant features (single unique value = useless) ─────────
constant_train = [c for c in feat_cols if df_train[c].nunique() <= 1]
constant_test  = [c for c in feat_cols if df_test[c].nunique()  <= 1]
print(f"Constant features in train: {constant_train or 'none'}")
print(f"Constant features in test:  {constant_test  or 'none'}")

# Near-constant: >99.9% of values are the same
near_const_train = [c for c in feat_cols
                    if df_train[c].value_counts(normalize=True).iloc[0] > 0.999]
print(f"\nNear-constant features in train (>99.9% one value): {near_const_train or 'none'}")

In [ ]:
# ── Drop duplicates and constant features; log decisions ─────────────────────
cols_to_drop = list(set(constant_train + constant_test))

df_train_clean = df_train.drop_duplicates(subset=feat_cols).reset_index(drop=True)
df_test_clean  = df_test.drop_duplicates(subset=feat_cols).reset_index(drop=True)

if cols_to_drop:
    df_train_clean = df_train_clean.drop(columns=cols_to_drop)
    df_test_clean  = df_test_clean.drop(columns=cols_to_drop)

print(f"After deduplication:")
print(f"  Train: {len(df_train):,} → {len(df_train_clean):,} rows  "
      f"(removed {len(df_train) - len(df_train_clean):,} duplicates)")
print(f"  Test:  {len(df_test):,}  → {len(df_test_clean):,}  rows  "
      f"(removed {len(df_test) - len(df_test_clean):,} duplicates)")
if cols_to_drop:
    print(f"\nDropped constant columns: {cols_to_drop}")
else:
    print("\nNo constant columns dropped.")

**Hygiene findings — cleaned shapes: 266,739 × 67 train, 106,906 × 67 test:**

- **Duplicates removed:** 14,695 from train (5.2%), 17,790 from test (14.3%). Common in network data — attack scripts hitting identical parameters, or benign apps opening repeated identical connections. Higher rate in test reflects the more uniform pre-processed 2018 file.
- **10 constant columns dropped:** `Bwd Avg Bulk Rate/Bytes/Packets`, `Fwd Avg Bulk Rate/Bytes/Bulk`, `Bwd PSH Flags`, `Bwd URG Flags`, `Fwd URG Flags`, `CWE Flag Count` — all zero across every row. Known CICFlowMeter limitation: bulk-transfer heuristics rarely trigger in lab-simulated traffic. Zero-variance features add only noise to a model.
- **Near-constant but kept:** `ECE Flag Count` and `RST Flag Count` exceed the 99.9% threshold in training but vary in the test set — rare non-zero values may still carry signal.
- Cleaned frames saved as `train_clean.joblib` / `test_clean.joblib` to Drive.


In [ ]:
# ── Save cleaned frames to Drive ──────────────────────────────────────────────
from joblib import dump

dump(df_train_clean, os.path.join(DATA_DIR, 'train_clean.joblib'))
dump(df_test_clean,  os.path.join(DATA_DIR, 'test_clean.joblib'))
print("Saved train_clean.joblib and test_clean.joblib to Drive.")

---
## §2 — Exploratory Data Analysis

Understand training and test distributions before feature engineering. Covers:
- **§2.1** Class distribution and the class-imbalance problem
- **§2.2** Feature distributions for key network-flow statistics
- **§2.3** Missing values verification
- **§2.4** Outlier analysis
- **§2.5** Temporal-feature analysis
- **§2.6** Cross-tabulation and group-by analysis
- **§2.7** Correlation analysis — method choice and justification (Spearman)
- **§2.8** Pre- vs post-balancing: visualising the 'symmetry' trade-off


In [ ]:
# §2 setup — reload cleaned frames if needed, define FIGURES_DIR
import os, pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from joblib import load as jload

try:
    df_train_clean
except NameError:
    df_train_clean = jload(os.path.join(DATA_DIR, 'train_clean.joblib'))
    df_test_clean  = jload(os.path.join(DATA_DIR, 'test_clean.joblib'))
    print('Reloaded cleaned frames from Drive.')

feat_cols_clean = [c for c in df_train_clean.columns if c != 'Label']
mask_benign = df_train_clean['Label'].str.strip().str.upper() == 'BENIGN'
print(f'Train: {df_train_clean.shape}, Test: {df_test_clean.shape}, Features: {len(feat_cols_clean)}')

FIGURES_DIR = str(pathlib.Path(DATA_DIR).parent / 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)
sns.set_theme(style='whitegrid', palette='tab10')


### §2.1 — Class Distribution and Imbalance Analysis


In [ ]:
def binary_counts(df):
    b = df['Label'].apply(lambda x: 'BENIGN' if str(x).strip().upper()=='BENIGN' else 'ATTACK')
    return b.value_counts().reindex(['BENIGN','ATTACK'], fill_value=0)

train_bc = binary_counts(df_train_clean)
test_bc  = binary_counts(df_test_clean)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (name, bc) in zip(axes, [('Train (CIC-IDS2017)', train_bc),
                                   ('Test (CSE-CIC-IDS2018)', test_bc)]):
    bars = ax.bar(bc.index, bc.values, color=['steelblue','tomato'],
                  edgecolor='white', linewidth=1.2)
    ax.set_title(name, fontsize=12)
    ax.set_ylabel('Row count')
    for bar, (lbl, val) in zip(bars, bc.items()):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+300,
                f'{val:,}\n({val/bc.sum()*100:.1f}%)',
                ha='center', va='bottom', fontsize=10)

plt.suptitle('Class distribution — before balancing (real prevalence)', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'class_distribution_raw.png'), dpi=150, bbox_inches='tight')
plt.show()

print('\n--- Train: attack-type breakdown ---')
print(df_train_clean['Label'].value_counts().to_string())
print('\n--- Test: attack-type breakdown ---')
print(df_test_clean['Label'].value_counts().to_string())


**Class distribution** *(Figure 1)*: 218,453 BENIGN vs 48,286 attack in train (**81.9% benign**); 89,937 vs 16,969 in test (**84.1% benign**). Both reflect typical enterprise network conditions.

**Attack-type drift — the core threat to the paper's conclusions:**

| Attack type | Train | Test | Change |
|-------------|-------|------|--------|
| Web Attack - XSS | 70 | Brute Force - XSS: **10,489** | ~150× amplification |
| PortScan | **14,444** | 0 | Disappears entirely |
| DDoS (generic) | 12,732 | HOIC: 3,415 / LOIC-UDP: 861 / LOIC-HTTP: 216 | Renamed & fragmented |
| Bot | 199 | 34 | Shrinks |
| Heartbleed | **1** | 0 | Effectively unlearnable |
| Infiltration | **6** | 0 | Effectively unlearnable |

A model trained on 70 XSS flows that then faces 10,489 XSS flows at test time is experiencing **concept drift**, not classical overfitting. The paper conflates the two — Phase 8.1 provides a direct test.

The authors' 1:1 downsampling also means accuracy on the balanced test set does not reflect deployment where 84% of inputs are benign. Quantified in §2.8.


### §2.2 — Feature Distributions


In [ ]:
KEY_FEATURES = [
    'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets',
    'Flow Bytes/s', 'Fwd Packet Length Mean', 'Bwd Packet Length Mean',
    'Fwd IAT Mean', 'Bwd IAT Mean', 'Flow IAT Mean',
]
KEY_FEATURES = [f for f in KEY_FEATURES if f in feat_cols_clean]

ncols = 3
nrows = (len(KEY_FEATURES) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(16, 4 * nrows))
axes = axes.flatten()

for ax, feat in zip(axes, KEY_FEATURES):
    clip_val = df_train_clean[feat].quantile(0.99)
    for lbl, mask, color in [
        ('BENIGN', mask_benign, 'steelblue'),
        ('ATTACK', ~mask_benign, 'tomato')
    ]:
        ax.hist(df_train_clean.loc[mask, feat].clip(upper=clip_val),
                bins=50, alpha=0.5, color=color, label=lbl, density=True)
    ax.set_title(feat, fontsize=9)
    ax.tick_params(labelsize=7)

axes[0].legend(fontsize=9)
for ax in axes[len(KEY_FEATURES):]:
    ax.set_visible(False)

plt.suptitle('Feature distributions by class — train set (99th-percentile clipped)', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'feature_distributions.png'), dpi=150, bbox_inches='tight')
plt.show()


**Feature distributions** *(Figure 2)*:

- All features are **heavily right-skewed** — most flows are short with few packets, with a heavy tail of high-volume flows. This violates Pearson's normality assumption and motivates Spearman (§2.7).
- **`Flow Duration`** (attack) is bimodal: spike near zero (fast DoS floods) + mass at ~10⁸ µs (slowloris/Hulk keeping connections open). Two completely different attack mechanics in one binary label.
- **`Bwd Packet Length Mean`** (attack) is also bimodal: spike at 0 (Slowhttptest — no server response) + mass at 1,500–2,000 bytes (DDoS response packets).
- **`Flow Bytes/s`** shows negative values — a CICFlowMeter edge-case artifact in this dataset.
- The per-feature overlap between benign and attack explains why no single feature suffices; the paper's 11-feature selection (§3) targets the most discriminative *combination*.


### §2.3 — Missing Values Verification


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, (name, df) in zip(axes, [('Train', df_train_clean), ('Test', df_test_clean)]):
    nan_counts = df[feat_cols_clean].isnull().sum()
    nan_nz = nan_counts[nan_counts > 0]
    if nan_nz.empty:
        ax.text(0.5, 0.5, 'No missing values\n(cleaned in §1)',
                ha='center', va='center', transform=ax.transAxes,
                fontsize=13, color='seagreen')
    else:
        nan_nz.sort_values(ascending=False).plot(kind='bar', ax=ax, color='tomato')
        ax.set_ylabel('NaN count')
    ax.set_title(f'{name} — NaN per feature')

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'missing_values.png'), dpi=150, bbox_inches='tight')
plt.show()

for name, df in [('Train', df_train_clean), ('Test', df_test_clean)]:
    numeric = df[feat_cols_clean].select_dtypes(include='number')
    inf_count = np.isinf(numeric).sum().sum()
    print(f'{name} — inf values remaining: {inf_count}')


### §2.4 — Outlier Analysis


In [ ]:
print('Outlier rate (IQR method, >Q3 + 1.5*IQR) — train set:\n')
outlier_summary = {}
for feat in KEY_FEATURES:
    q1, q3 = df_train_clean[feat].quantile([0.25, 0.75])
    upper = q3 + 1.5 * (q3 - q1)
    n_out = int((df_train_clean[feat] > upper).sum())
    outlier_summary[feat] = n_out
    print(f'  {feat:<35}: {n_out:>6,}  ({n_out/len(df_train_clean)*100:.1f}%)')

top6 = sorted(outlier_summary, key=outlier_summary.get, reverse=True)[:6]
top6 = [f for f in top6 if f in df_train_clean.columns]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, feat in zip(axes.flatten(), top6):
    cap = df_train_clean[feat].quantile(0.999)
    data = [
        df_train_clean.loc[mask_benign, feat].clip(upper=cap).values,
        df_train_clean.loc[~mask_benign, feat].clip(upper=cap).values,
    ]
    bp = ax.boxplot(data, tick_labels=['Benign', 'Attack'], patch_artist=True, showfliers=False)
    for patch, color in zip(bp['boxes'], ['steelblue', 'tomato']):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)
    ax.set_title(feat, fontsize=9)
    ax.tick_params(labelsize=8)

plt.suptitle(
    'Box plots — highest-outlier features by class (99.9th-pct clipped, no fliers shown)',
    fontsize=11
)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'outlier_boxplots.png'), dpi=150, bbox_inches='tight')
plt.show()


**Outlier findings** *(Figure 4)*: IQR outlier rates of 5–20% per feature — expected, not a quality problem.

- **DoS/DDoS:** outliers in `Flow Bytes/s` and IAT — flooding at maximum rate.
- **PortScan:** extreme values in packet-length features (tiny probes) and very short `Flow Duration`.
- **Benign elephant flows:** genuine outliers in byte-count features (large file transfers, video streaming).
- **Counterintuitive:** BENIGN has a *higher* median `Flow Bytes/s` than ATTACK (~210k vs ~100k bytes/s). Slow DoS attacks (slowloris, Slowhttptest) have near-zero byte rates, pulling the attack median down. `Flow Bytes/s` alone is not a reliable attack indicator.

**Decision — do not clip outliers:** tree models (DT, RF) handle them via split thresholds; SVM/ANN inputs will be scaled in §3. Clipping would destroy the attack-fingerprint patterns.


### §2.5 — Temporal Feature Analysis


In [ ]:
TEMPORAL_FEATS = [f for f in
    ['Flow Duration', 'Fwd IAT Mean', 'Bwd IAT Mean', 'Flow IAT Mean']
    if f in feat_cols_clean]

top10_labels = df_train_clean['Label'].value_counts().head(10).index.tolist()
df_sub = df_train_clean[df_train_clean['Label'].isin(top10_labels)]

fig, axes = plt.subplots(len(TEMPORAL_FEATS), 1, figsize=(13, 4 * len(TEMPORAL_FEATS)))
if len(TEMPORAL_FEATS) == 1:
    axes = [axes]

for ax, feat in zip(axes, TEMPORAL_FEATS):
    medians = df_sub.groupby('Label')[feat].median().sort_values()
    colors = ['steelblue' if 'BENIGN' in str(l).upper() else 'tomato'
              for l in medians.index]
    ax.barh(medians.index, medians.values, color=colors)
    ax.set_title(f'Median {feat} by class', fontsize=11)
    ax.set_xlabel(feat)
    ax.tick_params(labelsize=9)

plt.suptitle('Temporal features by attack class — top 10 classes, train set', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'temporal_features_by_class.png'), dpi=150, bbox_inches='tight')
plt.show()


**Temporal features by class** *(Figure 5)* — attack fingerprints align with known network behaviour:

- **DoS/DDoS** (GoldenEye, Hulk, LOIC/HOIC): near-zero IAT + short duration — flooding at maximum rate.
- **Slowloris / Slowhttptest:** longest flow durations (~10⁸ µs) — intentionally kept alive; near-zero Bwd IAT because server barely responds.
- **DoS Hulk:** high Fwd IAT despite being a DoS attack — Hulk sends valid HTTP GETs with pauses, not raw packet flooding.
- **PortScan:** near-zero duration and IAT (connection attempts fail immediately).
- **FTP-Patator / SSH-Patator:** moderate duration (full authentication protocol exchange per attempt).
- **BENIGN:** highest IAT variance — irregular, human-paced behaviour mixing DNS queries and streaming.

Note: DoS attacks dominate the axis scale — BENIGN, Bot, and PortScan bars are nearly invisible in Figure 5. This is a scale limitation, not a data problem.

**Timestamp excluded:** including calendar timestamps would teach the model '2017 = benign, 2018 = attack' (time leakage). IAT and Duration capture structural flow timing and are safe.


### §2.6 — Cross-tabulation and Group-by Analysis


In [ ]:
SUMMARY_FEATS = [f for f in
    ['Flow Duration','Total Fwd Packets','Total Backward Packets',
     'Flow Bytes/s','Fwd Packet Length Mean','Bwd Packet Length Mean']
    if f in feat_cols_clean]

top8 = df_train_clean['Label'].value_counts().head(8).index.tolist()
group_stats = (
    df_train_clean[df_train_clean['Label'].isin(top8)]
    .groupby('Label')[SUMMARY_FEATS]
    .median()
    .round(2)
)
print('Median feature values by class (top 8, train set):')
print(group_stats.to_string())
print()

if 'Total Fwd Packets' in feat_cols_clean and 'Total Backward Packets' in feat_cols_clean:
    ratio = (df_train_clean['Total Backward Packets'] /
             (df_train_clean['Total Fwd Packets'] + 1e-6)).clip(0, 100)
    ratio_by_class = (
        df_train_clean[df_train_clean['Label'].isin(top8)]
        .assign(_ratio=ratio)
        .groupby('Label')['_ratio']
        .median()
        .sort_values()
    )
    print('Median Bwd/Fwd packet ratio by class:')
    print('  (0 = purely unidirectional DoS;  ~1 = balanced bidirectional traffic)')
    print(ratio_by_class.to_string())


**Bwd/Fwd packet ratio by class** — confirms known network-security patterns:

| Class | Median Bwd/Fwd | Note |
|-------|---------------|------|
| DoS Slowhttptest | **0.000** | Purely unidirectional — server never responds |
| DoS slowloris | 0.214 | Mostly unidirectional — connection starved |
| DoS GoldenEye | 0.625 | HTTP requests get some responses before slowdown |
| DDoS | 0.750 | Victim responds to a fraction of the flood |
| DoS Hulk | 0.857 | Near-symmetric — server tries to answer valid HTTP GETs |
| BENIGN | **1.000** | Perfectly bidirectional (TCP/HTTP/DNS) |
| PortScan | **1.000** | CICFlowMeter captures SYN+SYN-ACK → appears symmetric |
| FTP-Patator | **1.667** | *Server-heavy:* multiple 530 failure messages per login attempt |

- **FTP-Patator > 1** is the most unusual finding: attacker-initiated traffic generates *more server-side* packets than client-side — the FTP server sends banners, challenges, and error codes for every failed attempt.
- **PortScan = 1** is counterintuitive: CICFlowMeter captures completed SYN+SYN-ACK flows, not raw SYN probes — so packet-direction features alone won't easily distinguish scans from benign traffic.


### §2.7 — Correlation Analysis — Method Choice and Justification


**Why Spearman (not Pearson or Kendall):**

| Method | Key assumption | Verdict for this dataset |
|--------|---------------|-------------------------|
| **Pearson** | Linear relationship; normally distributed, homoscedastic data | ❌ Network features follow power-law distributions; outliers are attack signals, not errors; relationship is monotonic but not strictly linear |
| **Kendall** | Rank-based; no normality assumption; robust to outliers | ✅ Correct assumptions, but **O(n²)** computation — impractical for ~267k rows and 66 features |
| **Spearman** | Rank-based; no normality assumption; robust to outliers | ✅ Captures monotonic relationships; **O(n log n)**; correct and efficient for this scale |

**Practical vs. statistical significance:** With ~267k rows, virtually every non-zero correlation is statistically significant (p < 0.001). What matters is *practical* significance: |r| > 0.90 identifies feature pairs carrying essentially redundant information — candidates for removal in §3 feature selection.


In [ ]:
# Spearman on a 10k-row subsample — fast and representative
_CORR_N = 10_000
df_corr_sample = df_train_clean[feat_cols_clean].sample(
    min(_CORR_N, len(df_train_clean)), random_state=SEED
)
print(f'Computing Spearman matrix on {len(df_corr_sample):,} rows x {len(feat_cols_clean)} features...')
corr_matrix = df_corr_sample.corr(method='spearman')
print('Done.')

# Pairs with |r| > 0.90 — redundant features
high_pairs = []
cols = corr_matrix.columns.tolist()
for i in range(len(cols)):
    for j in range(i+1, len(cols)):
        r = float(corr_matrix.iloc[i, j])
        if abs(r) > 0.90:
            high_pairs.append((abs(r), r, cols[i], cols[j]))
high_pairs.sort(reverse=True)

print(f'\nFeature pairs |Spearman r| > 0.90  ({len(high_pairs)} total — top 20 shown):')
for _, r, c1, c2 in high_pairs[:20]:
    print(f'  r={r:+.3f}  {c1}  <->  {c2}')


In [ ]:
# Spearman heatmap — top 30 highest-variance features, hierarchically clustered
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import squareform

top30 = df_corr_sample.var().nlargest(30).index.tolist()
sub_corr = corr_matrix.loc[top30, top30]

try:
    dist_mat = (1 - sub_corr.abs()).clip(lower=0)
    np.fill_diagonal(dist_mat.values, 0.0)
    link = linkage(squareform(dist_mat.values), method='average')
    order = leaves_list(link)
    sub_corr = sub_corr.iloc[order, order]
except Exception as e:
    print(f'Hierarchical clustering skipped ({e}); using original order.')

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(sub_corr, ax=ax, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.3, cbar_kws={'label': 'Spearman r'},
            xticklabels=True, yticklabels=True)
ax.tick_params(axis='x', labelrotation=45, labelsize=7)
ax.tick_params(axis='y', labelrotation=0, labelsize=7)
ax.set_title('Spearman correlation — top 30 highest-variance features (train set)', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'spearman_correlation_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()


**Spearman correlation** *(Figure 6)*: 99 pairs with |r| > 0.90. Ten pairs are r = 1.000 (mathematically identical):

| Pair (r = 1.000) | Reason |
|-----------------|--------|
| `Subflow Fwd/Bwd Packets/Bytes` ↔ `Total Fwd/Bwd Packets/Length` (4 pairs) | CICFlowMeter: Subflow = Total for single-subflow flows |
| `Avg Fwd/Bwd Segment Size` ↔ `Fwd/Bwd Packet Length Mean` (2 pairs) | Two names for the same computation |
| `Packet Length Std` ↔ `Packet Length Variance` | Mathematical identity: Var = Std² |
| `Idle Max` ↔ `Idle Mean` | Near-constant idle times |
| `Fwd PSH Flags` ↔ `SYN Flag Count` | CIC capture artefact |
| `ECE Flag Count` ↔ `RST Flag Count` | CIC capture artefact |

**Three redundancy clusters (visible in Figure 6):**
1. **Subflow cluster:** four subflow features are exact duplicates of four total-flow features → all four should be dropped.
2. **Packet length / segment size:** `Avg Segment Size` = `Packet Length Mean`; `Packet Length Variance` = `Packet Length Std²` → keep one per pair.
3. **IAT cluster:** `Bwd/Fwd/Flow IAT Max/Mean/Total` all r ≥ 0.997 with each other → one representative suffices.

The flag artefacts (PSH↔SYN, ECE↔RST) are CIC-specific — if these correlations break in a different network capture, a model relying on them will degrade, contributing to the cross-dataset performance drop.

This directly motivates the authors' **66 → 11 feature reduction** reproduced in §3.


### §2.8 — Pre- vs Post-Balancing: the 'Symmetry' Trade-off


In [ ]:
df_bin = df_train_clean.assign(
    _binary=df_train_clean['Label'].apply(
        lambda x: 'BENIGN' if str(x).strip().upper()=='BENIGN' else 'ATTACK'
    )
)
before = df_bin['_binary'].value_counts().reindex(['BENIGN','ATTACK'], fill_value=0)

n_min = int(before.min())
balanced = pd.concat([
    df_bin[df_bin['_binary']==cls].sample(n=n_min, random_state=SEED)
    for cls in ['BENIGN','ATTACK']
])
after = balanced['_binary'].value_counts().reindex(['BENIGN','ATTACK'])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (title, bc) in zip(axes, [
    ('Before balancing — real prevalence', before),
    ("After 1:1 downsampling (paper's 'symmetry')", after)
]):
    bars = ax.bar(bc.index, bc.values, color=['steelblue','tomato'],
                  edgecolor='white', linewidth=1.2)
    ax.set_title(title, fontsize=11)
    ax.set_ylabel('Row count')
    total = bc.sum()
    for bar, val in zip(bars, bc.values):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+200,
                f'{val:,}\n({val/total*100:.1f}%)',
                ha='center', va='bottom', fontsize=10)

plt.suptitle("Class distribution: real prevalence vs. paper's 1:1 'symmetry'", fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'class_balancing_effect.png'), dpi=150, bbox_inches='tight')
plt.show()

discarded = int(before.sum() - after.sum())
print(f'Rows discarded by 1:1 downsampling: {discarded:,}')
print(f'Training data retained: {after.sum()/before.sum()*100:.1f}%')


**The real cost of 1:1 'symmetry'** *(Figure 7)*: keeping only 48,286 of 218,453 BENIGN rows means **170,167 rows discarded — 63.8% of all training data**. Only 36.2% is used.

- **Metric inflation:** accuracy on a 1:1 balanced test set is systematically higher than on a deployment-realistic 84%-benign set. The paper's reported ~96–99% in-distribution accuracy is an upper bound, not a deployment estimate.
- **Data waste:** the model sees a less representative sample of legitimate traffic — rare-but-legitimate flows that fall in the discarded majority may become false positives.
- **Alternatives not explored:** class-weighted loss, SMOTE (synthetic oversampling), or threshold tuning — all address imbalance without discarding 63.8% of the data. The paper's 'symmetry' framing elevates a pragmatic compromise to a principle it doesn't warrant.

Phase 8.2 re-evaluates all models on the real-prevalence test split to measure the accuracy gap directly.


---
## §3 — Feature Engineering

Reproduce the authors' pipeline: cleaning → balancing → binary relabeling → encoding → scaling → feature creation → feature selection.

---
## §4 — Model Training

Train DT, RF, SVM, NB, ANN, DNN with GridSearchCV (k=5). Save best models to Drive.

---
## §5 — Evaluation & Reproduction Check

In-distribution evaluation (reproduce Tables 4–6). Progressive evaluation on CSE-CIC-IDS2018 (reproduce Table 7). Side-by-side comparison with paper's numbers.

---
## §6 — Error Analysis

Misclassified examples (FPs and FNs) on the progressive test set. Patterns in errors. Cybersecurity implications.

---
## §7 — Executive Summary

*(Filled after all analysis is complete.)*

---
## §8 — Summing It Up

*(Filled after all analysis is complete.)*